# AgriTwin — Field-Robust Crop Disease Model (Colab / T4 GPU)

Cross-domain training pipeline: lab-only baseline → PlantDoc-only → mixed → two-stage adaptation, plus an optional ReVit-derived ExG ablation. Downloads PlantDoc, builds a deterministic manifest, trains the experiments, generates Grad-CAM figures, benchmarks inference, and saves the deployable `model.pt` + `metadata.json` back to Google Drive.

## One-time setup (do this on your PC)
Copy these folders into `MyDrive/AgriTwin/` in Google Drive:

- `ml/crop_health/` (the 1,900 PlantVillage images and the training scripts)
- `backend/app/ml/` (includes `crop_health/labels.py`, `gradcam.py`, `exg.py` and the current `artifacts/crop_health/metadata.json`)

Then set **Runtime → Change runtime type → T4 GPU** and run the cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = "/content/drive/MyDrive/AgriTwin"  # <-- edit if yours lives elsewhere
LOCAL_ROOT = "/content/agritwin"

import shutil
from pathlib import Path

drive_root = Path(DRIVE_PROJECT)
for rel in ["ml/crop_health", "backend/app/ml"]:
    src = drive_root / rel
    assert src.exists(), f"Missing {src} — copy it from your local project first (see above)"
    dst = Path(LOCAL_ROOT) / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

pv = Path(LOCAL_ROOT) / "ml/crop_health/data/plantvillage_subset"
print("Copied project to", LOCAL_ROOT)
print("PlantVillage images:", len(list(pv.glob("*/*"))))

In [ ]:
# Download PlantDoc. git clone can fail with an early EOF on some networks,
# so fall back to verified per-file downloads from the GitHub tree API.
import subprocess, urllib.parse, concurrent.futures as cf
import requests

pd_dir = Path(LOCAL_ROOT) / "ml/crop_health/data/plantdoc"
if not (pd_dir / "train").exists():
    pd_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/pratikkayal/PlantDoc-Dataset.git", str(pd_dir)],
            check=True, timeout=900,
        )
    except Exception as exc:
        print("git clone failed:", exc, "-> using per-file downloader")
        tree = requests.get(
            "https://api.github.com/repos/pratikkayal/PlantDoc-Dataset/git/trees/master?recursive=1",
            timeout=90,
        ).json()
        items = [x for x in tree["tree"]
                 if x.get("type") == "blob"
                 and x["path"].startswith(("train/", "test/"))
                 and x["path"].lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]

        def get_one(item):
            target = pd_dir / item["path"]
            if target.exists() and target.stat().st_size == item.get("size"):
                return True
            url = "https://raw.githubusercontent.com/pratikkayal/PlantDoc-Dataset/master/" \
                  + urllib.parse.quote(item["path"], safe="/")
            for attempt in range(5):
                try:
                    r = requests.get(url, timeout=90)
                    if r.status_code == 200 and len(r.content) == item.get("size"):
                        target.parent.mkdir(parents=True, exist_ok=True)
                        target.write_bytes(r.content)
                        return True
                except Exception:
                    pass
            return False

        with cf.ThreadPoolExecutor(max_workers=16) as pool:
            ok = sum(pool.map(get_one, items))
        print(f"Downloaded {ok}/{len(items)} images")
        assert ok == len(items), "Some PlantDoc files failed — re-run this cell"

folders = sorted(p.name for p in (pd_dir / "train").iterdir() if p.is_dir())
print(f"PlantDoc: {len(folders)} classes, e.g. {folders[:4]}")

In [ ]:
# (Optional) Upload your failed Google images as class folders into
# /content/agritwin/ml/crop_health/data/web_images/ before this cell,
# then add  --web-dir /content/agritwin/ml/crop_health/data/web_images  below.
import os, sys
os.chdir(LOCAL_ROOT)
subprocess.run(
    [sys.executable, "ml/crop_health/training/prepare_field_dataset.py",
     "--plantdoc-dir", "ml/crop_health/data/plantdoc"],
    check=True,
)

In [ ]:
# Train all four cross-domain experiments (~20-40 min on a T4).
# --amp enables FP16 mixed precision on the T4 for a large speedup.
subprocess.run(
    [sys.executable, "ml/crop_health/training/train_field_robust.py",
     "--mode", "all",
     "--epochs", "12",
     "--deploy-experiment", "mixed",
     "--batch-size", "32",
     "--amp"],
    check=True,
)

In [ ]:
# Optional E5 ablation: ReVit-derived ExG background suppression.
# Keep it in a separate artifact dir so the main deployable model is untouched.
RUN_EXG_ABLATION = False  # set True after E1-E4 have completed
if RUN_EXG_ABLATION:
    subprocess.run(
        [sys.executable, "ml/crop_health/training/train_field_robust.py",
         "--mode", "mixed",
         "--epochs", "12",
         "--batch-size", "32",
         "--amp",
         "--exg-suppression", "--exg-strength", "0.8",
         "--artifact-dir", "backend/app/ml/artifacts/crop_health_exg",
         "--eval-dir", "ml/crop_health/evaluation/exg"],
        check=True,
    )

In [ ]:
# Hardware efficiency numbers for the paper.
subprocess.run(
    [sys.executable, "ml/crop_health/evaluation/benchmark.py",
     "--architecture", "all"],
    check=True,
)

In [ ]:
# Copy the deployable artifact + evaluation back to Google Drive.
out = drive_root / "artifacts_from_colab"
out.mkdir(parents=True, exist_ok=True)

for rel in [
    "backend/app/ml/artifacts/crop_health/model.pt",
    "backend/app/ml/artifacts/crop_health/metadata.json",
    "ml/crop_health/evaluation/cross_domain_results.json",
    "ml/crop_health/evaluation/benchmark_results.json",
]:
    src = Path(LOCAL_ROOT) / rel
    dst = out / Path(rel).name
    shutil.copy2(src, dst)
    print("Saved", dst)

gc = Path(LOCAL_ROOT) / "ml/crop_health/evaluation/gradcam"
if gc.exists():
    shutil.copytree(gc, out / "gradcam", dirs_exist_ok=True)
    print("Saved Grad-CAM figures to", out / "gradcam")

In [ ]:
# Print the cross-domain summary table for your paper.
import json

results = json.load(open(out / "cross_domain_results.json"))
for row in results["summary"]:
    print(row)

### Next step
Copy `model.pt` and `metadata.json` from `MyDrive/AgriTwin/artifacts_from_colab/` into your local project at `backend/app/ml/artifacts/crop_health/`, then restart the FastAPI backend. Keep `cross_domain_results.json` and `benchmark_results.json` — they are the sources for the paper's results and efficiency tables.